!pip install findspark

### NY Parking Violations

### NY Parking Violations

In [57]:
# Checking if pyspark is installed properly and its path is right

import pyspark
import findspark

findspark.init()

In [58]:
!pip install findspark
!pip install scipy
!pip install pandas


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [59]:
# Importing all the required libraries
import scipy
from scipy import spatial
import numpy as np
import pandas as pd
import sys
import pyspark 
from pyspark.sql import Row
from pyspark.sql import functions as pysf
from pyspark.sql.window import Window
from pyspark.sql.functions import desc, when, count, col, sum, regexp_replace,expr, avg, first
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import SparkSession 
from pyspark.sql import functions as Func
from pyspark.sql.types import IntegerType
from pyspark import SparkContext, SparkConf
from functools import reduce

In [60]:
# Creating Spark Session with the name 'ECC Assignment 2'
# Allocating for 4 cores
spark = SparkSession.builder\
        .master("local[4]")\
        .appName("Assgn_2")\
        .getOrCreate()

In [61]:
# Loading the Parking_Violations_Issued_-_Fiscal_Year_2023_20231113 from the given link
# The dataset on the link was updated on Nov 15, 2023, under the '2024' and it was blank
# Therefore used the just previous version of the dataset

parking_df = spark.read.csv("pvqr-7yc4.csv", header=True, inferSchema=True)

In [62]:
parking_df.show()

+--------------+--------+------------------+----------+-------------------+--------------+-----------------+------------+--------------+------------+------------+------------+-----------------------+------------------+------------------+---------------+-----------+--------------+------------+--------------+-------------------+----------------+---------------------------------+------------+------------------+-------------------+-------------------+-----------+------------+--------------------+----------------------+--------------------+------------------+-------------+--------------------+------------+------------+--------------+-------------------+---------------------+---------------------------------+-----------------+------------------------+
|summons_number|plate_id|registration_state|plate_type|         issue_date|violation_code|vehicle_body_type|vehicle_make|issuing_agency|street_code1|street_code2|street_code3|vehicle_expiration_date|violation_location|violation_precinct|issuer_

In [63]:
from pyspark.sql.functions import col, count, when
from functools import reduce

# Specify the number of threads
num_threads = 4

# Split columns into groups
columns_per_thread = [parking_df.columns[i::num_threads] for i in range(num_threads)]

# Count nulls in parallel
counts_per_thread = []

for i, columns in enumerate(columns_per_thread):
    print(f"Processing thread {i + 1}/{num_threads}...")

    # Count nulls for the current set of columns
    counts = parking_df.select([count(when(col(c).isNull(), c)).alias(c) for c in columns]).collect()

    # Append the result to the list
    counts_per_thread.append(counts)

Processing thread 1/4...
Processing thread 2/4...
Processing thread 3/4...
Processing thread 4/4...


In [64]:
total_counts = reduce(lambda x, y: x + y, counts_per_thread)

total_counts

[Row(summons_number=0, issue_date=0, issuing_agency=0, vehicle_expiration_date=0, issuer_code=0, time_first_observed=880, street_name=2, sub_division=2, to_hours_in_effect=0, meter_number=0, no_standing_or_stopping_violation=1000),
 Row(plate_id=0, violation_code=0, street_code1=0, violation_location=11, issuer_command=0, violation_county=154, intersecting_street=813, violation_legal_code=1000, vehicle_color=117, feet_from_curb=0, hydrant_violation=1000),
 Row(registration_state=0, vehicle_body_type=43, street_code2=0, violation_precinct=0, issuer_squad=0, violation_in_front_of_or_opposite=150, date_first_observed=0, days_parking_in_effect=0, unregistered_vehicle=0, violation_post_code=1000, double_parking_violation=1000),
 Row(plate_type=0, vehicle_make=30, street_code3=0, issuer_precinct=0, violation_time=1, house_number=203, law_section=0, from_hours_in_effect=0, vehicle_year=0, violation_description=1000)]

In [65]:
# Above, I tried filtering out the dataset, however wasn't able to get it in a tabular format

#### Part 1

##### Question 1

When are tickets most likely to be issued?

In [66]:
# Creating a view which will come handy in querying the dataset
parking_df.createTempView("temp")

# Wrote a SQL query to count the frequency of each violation time
part1 = spark.sql('''SELECT `violation_time`, COUNT(*) AS count FROM temp 
            GROUP BY `violation_time` 
            ORDER BY count DESC LIMIT 30''')# Displaying the top 30 of them

part1.show(30)

+--------------+-----+
|violation_time|count|
+--------------+-----+
|         0540P|    8|
|         0435P|    6|
|         0640P|    6|
|         1000A|    6|
|         1245P|    6|
|         0620P|    6|
|         0915P|    6|
|         0655P|    6|
|         0100A|    5|
|         1150P|    5|
|         0825P|    5|
|         0730P|    5|
|         0250A|    5|
|         0200A|    5|
|         0300P|    5|
|         1040P|    5|
|         0900P|    5|
|         0625P|    5|
|         1255P|    4|
|         0930P|    4|
|         0630P|    4|
|         0820A|    4|
|         0405P|    4|
|         0425P|    4|
|         0145A|    4|
|         1050P|    4|
|         0530P|    4|
|         0905P|    4|
|         0705A|    3|
|         1100P|    3|
+--------------+-----+



##### Question 2

What are the most common years and types of cars to be ticketed?

In [67]:
# Wrote a Spark SQL query to count the number of violations for each vehicle body type and year
# Also, the below query takes care of filtering missing and negative values
# It also cast Vehicle Year to integer data type

part2 = spark.sql('''SELECT `vehicle_body_type` AS `vehicle_type`, CAST(`vehicle_year` AS INT) AS `vehicle_year`, COUNT(*) AS `violation_count` FROM temp 
                        WHERE `vehicle_body_type` IS NOT NULL AND `vehicle_year` > 0 
                        GROUP BY `vehicle_body_type`, `vehicle_year` 
                        ORDER BY `violation_count` DESC''')
part2.show(20)# Displaying only top 20

+------------+------------+---------------+
|vehicle_type|vehicle_year|violation_count|
+------------+------------+---------------+
|        SUBN|        2022|             34|
|        SUBN|        2021|             33|
|         SDN|        2021|             30|
|         SDN|        2019|             28|
|         SDN|        2017|             27|
|         SDN|        2016|             27|
|        SUBN|        2020|             26|
|        SUBN|        2018|             22|
|        SUBN|        2017|             22|
|        SUBN|        2019|             22|
|         SDN|        2020|             20|
|         SDN|        2013|             20|
|        SUBN|        2013|             19|
|        SUBN|        2016|             18|
|         SDN|        2014|             17|
|        SUBN|        2015|             16|
|        SUBN|        2014|             16|
|         SDN|        2015|             15|
|        SUBN|        2023|             15|
|         SDN|        2010|     

##### Question 3

Where are tickets most commonly issued?

In [68]:
# Wrote a Spark SQL query to count the number of violations for each violation location

part3 = spark.sql('''SELECT COUNT(*) AS Number_of_tickets, `Violation_Location` FROM temp 
                    GROUP BY `Violation_Location` 
                    ORDER BY Number_of_tickets DESC''')
part3.show(15)# Displaying top 15 locations

# However, here the first row have NULL as a location

+-----------------+------------------+
|Number_of_tickets|Violation_Location|
+-----------------+------------------+
|               68|                 9|
|               49|               110|
|               43|                10|
|               39|                14|
|               37|                43|
|               34|               107|
|               34|                60|
|               28|                20|
|               28|                17|
|               28|                75|
|               27|                34|
|               27|                84|
|               27|                30|
|               26|               100|
|               24|                 1|
+-----------------+------------------+
only showing top 15 rows



In [69]:
# Wrote another Spark SQL query that takes care of the NULL location
part3_en = spark.sql('''SELECT COUNT(*) AS Number_of_tickets, `Violation_Location` FROM temp 
                    WHERE `Violation_Location` IS NOT NULL GROUP BY `Violation_Location` 
                    ORDER BY Number_of_tickets DESC''')
part3_en.show(15)# Displaying top 15 locations

+-----------------+------------------+
|Number_of_tickets|Violation_Location|
+-----------------+------------------+
|               68|                 9|
|               49|               110|
|               43|                10|
|               39|                14|
|               37|                43|
|               34|               107|
|               34|                60|
|               28|                17|
|               28|                20|
|               28|                75|
|               27|                84|
|               27|                34|
|               27|                30|
|               26|               100|
|               24|                 1|
+-----------------+------------------+
only showing top 15 rows



##### Question 4

Which color of the vehicle is most likely to get a ticket?

In [70]:
# Wrote a Spark SQL query to count the number of violations for each vehicle color
part4 = spark.sql('''SELECT `Vehicle_Color`, COUNT(*) AS count FROM temp 
                    GROUP BY `Vehicle_Color`
                    ORDER BY count DESC''')
part4.show(20)# Displaying top 20

+-------------+-----+
|Vehicle_Color|count|
+-------------+-----+
|          BLK|  140|
|         NULL|  117|
|           WH|   70|
|        WHITE|   67|
|         GRAY|   66|
|           GY|   66|
|         BLUE|   52|
|          RED|   48|
|          GRY|   47|
|           BK|   45|
|        BLACK|   41|
|          WHT|   32|
|        SILVE|   28|
|         GREY|   28|
|           BL|   22|
|          BLU|   16|
|           RD|   14|
|           WT|   11|
|           GR|    8|
|          TAN|    6|
+-------------+-----+
only showing top 20 rows



#### Part 2: KNN

##### Question 5

Given a Black vehicle parking illegally at 34510, 10030, 34050 (street codes). What is the probability that it will get an ticket?

In [71]:
# Creating a function to find the nearest cluster center to a given data point
def nearest_cluster(data_point, cluster_centers):
    # Using the scipy.spatial.distance.cdist function to calculate the pairwise distances (data points and cluster centers)
    distances = scipy.spatial.distance.cdist([data_point], cluster_centers, metric="sqeuclidean")
    nearest_cluster_id = np.argmin(distances)#find the index of the minimum distance
    return nearest_cluster_id

In [72]:
parking_df = parking_df.selectExpr("CAST(`Street_Code1` AS FLOAT) AS Street_Code1",
                   "CAST(`Street_Code2` AS FLOAT) AS Street_Code2",
                   "CAST(`Street_Code3` AS FLOAT) AS Street_Code3",
                   "`Vehicle_Color`")

vector_assembler = VectorAssembler(inputCols=["Street_Code1", "Street_Code2", "Street_Code3"],
                                   outputCol="features")

parking_df = vector_assembler.transform(parking_df)

In [73]:
kmeans = KMeans(k=5) # Taking 5 clusters for now
k_means_model = kmeans.fit(parking_df.select('features'))
park_clustered = k_means_model.transform(parking_df).cache()

In [74]:
park_clustered.show()

+------------+------------+------------+-------------+--------------------+----------+
|Street_Code1|Street_Code2|Street_Code3|Vehicle_Color|            features|prediction|
+------------+------------+------------+-------------+--------------------+----------+
|         0.0|         0.0|         0.0|         BLUE|           (3,[],[])|         0|
|     17870.0|     25390.0|     32670.0|         GRAY|[17870.0,25390.0,...|         1|
|     17870.0|     25390.0|     32670.0|         GRAY|[17870.0,25390.0,...|         1|
|     12690.0|     41700.0|     61090.0|        WHITE|[12690.0,41700.0,...|         1|
|     12690.0|     41700.0|     61090.0|        WHITE|[12690.0,41700.0,...|         1|
|         0.0|         0.0|         0.0|           BK|           (3,[],[])|         0|
|      8690.0|     21690.0|     21740.0|         NULL|[8690.0,21690.0,2...|         0|
|         0.0|     61090.0|         0.0|          GRY|   [0.0,61090.0,0.0]|         4|
|         0.0|         0.0|         0.0|   

In [75]:
veh_black = ['BLK', 'BK.', 'BCK', 'BK', 'BLK.', 'Black', 'BC', 'BLAC', 'BK/', 'B LAC']
prob_veh_black = park_clustered.groupBy('prediction').agg(
    count(when(col('Vehicle_Color').isin(veh_black), 1)).alias('Count'),
    count('Vehicle_Color').alias('Total_Cars')
).orderBy('prediction')
prob_black = prob_veh_black.select(
    'prediction',
    'Count',
    'Total_Cars',
    (col('Count') / col('Total_Cars')).alias('Probability')
)

In [76]:
prob_black.show()

+----------+-----+----------+-------------------+
|prediction|Count|Total_Cars|        Probability|
+----------+-----+----------+-------------------+
|         0|   51|       264|0.19318181818181818|
|         1|   39|       199|0.19597989949748743|
|         2|   44|       166|0.26506024096385544|
|         3|   35|       175|                0.2|
|         4|   16|        79|0.20253164556962025|
+----------+-----+----------+-------------------+



In [91]:
cluster_centers = np.array(k_means_model.clusterCenters()).astype(float)
street_codes = np.array([34510.0, 10030.0, 34050.0])
cluster_id = int(nearest_cluster(street_codes, cluster_centers))

print('Cluster id for Street Code (34510, 10030, 34050) is:', cluster_id)

Cluster id for Street Code (34510, 10030, 34050) is: 3


In [95]:
# Displaying the required probability
prob_black.filter(col('prediction') == 3).show()

+----------+-----+----------+-----------+
|prediction|Count|Total_Cars|Probability|
+----------+-----+----------+-----------+
|         3|   35|       175|        0.2|
+----------+-----+----------+-----------+



### NBA

#### Question 1

For each pair of the players (A, B), we define the fear sore of A when facing B is the hit rate, such that B is closet defender when A is shooting. Based on the fear sore, for each player, please find out who is his "most unwanted defender".

In [79]:
nba_df = spark.read.csv("shot_logs.csv", header=True, inferSchema=True)

nba_df.printSchema()

root
 |-- GAME_ID: integer (nullable = true)
 |-- MATCHUP: string (nullable = true)
 |-- LOCATION: string (nullable = true)
 |-- W: string (nullable = true)
 |-- FINAL_MARGIN: integer (nullable = true)
 |-- SHOT_NUMBER: integer (nullable = true)
 |-- PERIOD: integer (nullable = true)
 |-- GAME_CLOCK: timestamp (nullable = true)
 |-- SHOT_CLOCK: double (nullable = true)
 |-- DRIBBLES: integer (nullable = true)
 |-- TOUCH_TIME: double (nullable = true)
 |-- SHOT_DIST: double (nullable = true)
 |-- PTS_TYPE: integer (nullable = true)
 |-- SHOT_RESULT: string (nullable = true)
 |-- CLOSEST_DEFENDER: string (nullable = true)
 |-- CLOSEST_DEFENDER_PLAYER_ID: integer (nullable = true)
 |-- CLOSE_DEF_DIST: double (nullable = true)
 |-- FGM: integer (nullable = true)
 |-- PTS: integer (nullable = true)
 |-- player_name: string (nullable = true)
 |-- player_id: integer (nullable = true)



In [40]:
nba_df.show()

+--------+--------------------+--------+---+------------+-----------+------+-------------------+----------+--------+----------+---------+--------+-----------+-----------------+--------------------------+--------------+---+---+-------------+---------+
| GAME_ID|             MATCHUP|LOCATION|  W|FINAL_MARGIN|SHOT_NUMBER|PERIOD|         GAME_CLOCK|SHOT_CLOCK|DRIBBLES|TOUCH_TIME|SHOT_DIST|PTS_TYPE|SHOT_RESULT| CLOSEST_DEFENDER|CLOSEST_DEFENDER_PLAYER_ID|CLOSE_DEF_DIST|FGM|PTS|  player_name|player_id|
+--------+--------------------+--------+---+------------+-----------+------+-------------------+----------+--------+----------+---------+--------+-----------+-----------------+--------------------------+--------------+---+---+-------------+---------+
|21400899|MAR 04, 2015 - CH...|       A|  W|          24|          1|     1|2024-04-07 01:09:00|      10.8|       2|       1.9|      7.7|       2|       made|   Anderson, Alan|                    101187|           1.3|  1|  2|brian roberts|   2031

In [41]:
# Grouping the dataframe by player and defender & aggregating the shot statistics
nba_grouped_df = nba_df.groupBy(col("player_id").alias("PlayerID"), col("CLOSEST_DEFENDER_PLAYER_ID").alias("DefenderID")) \
             .agg(sum(when(col("SHOT_RESULT") == "made", 1).otherwise(0)).alias("count1s"), 
                  sum(when(col("SHOT_RESULT") == "missed", 1).otherwise(0)).alias("count0s"))
nba_grouped_df.show()

+--------+----------+-------+-------+
|PlayerID|DefenderID|count1s|count0s|
+--------+----------+-------+-------+
|  203148|    101179|      0|      1|
|  202687|    201980|      1|      0|
|    2744|      1717|      0|      2|
|  203469|    202329|      1|      1|
|  201945|    202322|      0|      3|
|  202689|    202699|      6|      8|
|  202689|    203924|      1|      0|
|  203077|      2730|      1|      0|
|  203077|    201584|      2|      0|
|  202362|    201188|      2|      0|
|  202330|    201978|      2|      1|
|  202324|    203135|      1|      2|
|  203957|      2617|      1|      0|
|    2430|    203092|      3|      2|
|    2430|    200770|      1|      0|
|  202391|    202328|      0|      1|
|  101179|    202390|      0|      1|
|  201961|    203200|      1|      0|
|  202325|    203498|      1|      4|
|  203135|    201951|      1|      0|
+--------+----------+-------+-------+
only showing top 20 rows



In [42]:
nba_df.createTempView("nbatemp")

nba_grouped_df = spark.sql('''SELECT player_id AS PlayerID, CLOSEST_DEFENDER_PLAYER_ID AS DefenderID, 
                        SUM(CASE WHEN SHOT_RESULT = 'made' THEN 1 ELSE 0 END) AS count1s, 
                        SUM(CASE WHEN SHOT_RESULT = 'missed' THEN 1 ELSE 0 END) AS count0s FROM nbatemp 
                        GROUP BY player_id, CLOSEST_DEFENDER_PLAYER_ID''')
nba_grouped_df.show()

+--------+----------+-------+-------+
|PlayerID|DefenderID|count1s|count0s|
+--------+----------+-------+-------+
|  203148|    101179|      0|      1|
|  202687|    201980|      1|      0|
|    2744|      1717|      0|      2|
|  203469|    202329|      1|      1|
|  201945|    202322|      0|      3|
|  202689|    202699|      6|      8|
|  202689|    203924|      1|      0|
|  203077|      2730|      1|      0|
|  203077|    201584|      2|      0|
|  202362|    201188|      2|      0|
|  202330|    201978|      2|      1|
|  202324|    203135|      1|      2|
|  203957|      2617|      1|      0|
|    2430|    203092|      3|      2|
|    2430|    200770|      1|      0|
|  202391|    202328|      0|      1|
|  101179|    202390|      0|      1|
|  201961|    203200|      1|      0|
|  202325|    203498|      1|      4|
|  203135|    201951|      1|      0|
+--------+----------+-------+-------+
only showing top 20 rows



In [43]:
# Calculating the hit rate for each combination (player and defender)
nba_grouped_df = nba_grouped_df.withColumn("HitRate", expr("count1s / (count1s + count0s)"))

# Droping duplicates & filtering out records with "null" hit rates
nba_grouped_df = nba_grouped_df.dropDuplicates(["PlayerID", "HitRate"]).filter("HitRate is not null")
nba_grouped_df.show()

+--------+----------+-------+-------+-------------------+
|PlayerID|DefenderID|count1s|count0s|            HitRate|
+--------+----------+-------+-------+-------------------+
|  201945|    202322|      0|      3|                0.0|
|  203082|    203490|      0|      1|                0.0|
|  201600|    202324|      1|      3|               0.25|
|  201143|    202700|      7|      2| 0.7777777777777778|
|  202330|    201967|      2|      7| 0.2222222222222222|
|  201600|    201579|      2|      4| 0.3333333333333333|
|  202322|    201565|     13|     16| 0.4482758620689655|
|  101162|    201149|      8|      5| 0.6153846153846154|
|  201939|    201935|     10|      4| 0.7142857142857143|
|     977|    202718|      3|     11|0.21428571428571427|
|  202685|    201600|      6|      5| 0.5454545454545454|
|  201155|    101145|      1|      4|                0.2|
|  203458|    201941|      2|      3|                0.4|
|  202681|    202689|     11|     17|0.39285714285714285|
|  201228|    

In [44]:
# Grouping & selecting the min hit rate (each player)
nba_df_final = nba_grouped_df.groupBy("PlayerID").agg({"HitRate": "min"}).withColumn("HitRate", col("min(HitRate)"))

nba_grouped_df = nba_grouped_df.join(nba_df_final, ["PlayerID", "HitRate"]).drop("HitRate").select("PlayerID", "DefenderID")

nba_df_merged = nba_grouped_df.join(nba_df, (nba_grouped_df["PlayerID"] == nba_df["player_id"]) & (nba_grouped_df["DefenderID"] == nba_df["CLOSEST_DEFENDER_PLAYER_ID"]))

In [45]:
# Grouping & selecting relevant columns
nba_df_final = nba_df_merged.groupBy("PlayerID", "DefenderID") \
                    .agg(first("player_name").alias("Player Name"), first("CLOSEST_DEFENDER").alias("Most Unwanted Defender")) \
                    .orderBy("PlayerID") \
                    .limit(10)

nba_df_final.show()

+--------+----------+--------------+----------------------+
|PlayerID|DefenderID|   Player Name|Most Unwanted Defender|
+--------+----------+--------------+----------------------+
|     708|    203957| kevin garnett|           Exum, Dante|
|     977|    203937|   kobe bryant|        Anderson, Kyle|
|    1495|    203148|    tim duncan|        Roberts, Brian|
|    1713|      2037|  vince carter|       Crawford, Jamal|
|    1717|    201581|dirk nowtizski|           Hickson, JJ|
|    1718|    203079|   paul pierce|         Waiters, Dion|
|    1889|    201168|  andre miller|       Splitter, Tiago|
|    1890|    201229|  shawn marion|     Tolliver, Anthony|
|    1891|    201572|   jason terry|          Lopez, Brook|
|    1938|    203461| manu ginobili|      Bennett, Anthony|
+--------+----------+--------------+----------------------+



In [46]:
# Checkpoint
nba_df_copy = nba_df.select("*")

#### Question 2

For each player, we define the comfortable zone of shooting is a matrix of,

{SHOT DIST, CLOSE DEF DIST, SHOT CLOCK}

Please develop a Spark-based algorithm to classify each player's records into 4 comfortable zones. Considering the hit rate, which zone is the best for James Harden, Chris Paul, Stephen Curry, and Lebron James.

In [47]:
# Filtering out rows with null values and converting the shot result to binary
nba_df = nba_df.na.drop()
nba_df = nba_df.withColumn("SHOT_RESULT", when(col("SHOT_RESULT") == "made", 1).otherwise(0))

In [48]:
nba_df.show()

+--------+--------------------+--------+---+------------+-----------+------+-------------------+----------+--------+----------+---------+--------+-----------+-----------------+--------------------------+--------------+---+---+-------------+---------+
| GAME_ID|             MATCHUP|LOCATION|  W|FINAL_MARGIN|SHOT_NUMBER|PERIOD|         GAME_CLOCK|SHOT_CLOCK|DRIBBLES|TOUCH_TIME|SHOT_DIST|PTS_TYPE|SHOT_RESULT| CLOSEST_DEFENDER|CLOSEST_DEFENDER_PLAYER_ID|CLOSE_DEF_DIST|FGM|PTS|  player_name|player_id|
+--------+--------------------+--------+---+------------+-----------+------+-------------------+----------+--------+----------+---------+--------+-----------+-----------------+--------------------------+--------------+---+---+-------------+---------+
|21400899|MAR 04, 2015 - CH...|       A|  W|          24|          1|     1|2024-04-07 01:09:00|      10.8|       2|       1.9|      7.7|       2|          1|   Anderson, Alan|                    101187|           1.3|  1|  2|brian roberts|   2031

In [49]:
# Assignment of features
features = ["SHOT_DIST", "CLOSE_DEF_DIST", "SHOT_CLOCK"]

# converitng all the columns to float data type
nba_df = reduce(lambda df, col: df.withColumn(col, df[col].cast("float")), features, nba_df)
nba_df.show()

+--------+--------------------+--------+---+------------+-----------+------+-------------------+----------+--------+----------+---------+--------+-----------+-----------------+--------------------------+--------------+---+---+-------------+---------+
| GAME_ID|             MATCHUP|LOCATION|  W|FINAL_MARGIN|SHOT_NUMBER|PERIOD|         GAME_CLOCK|SHOT_CLOCK|DRIBBLES|TOUCH_TIME|SHOT_DIST|PTS_TYPE|SHOT_RESULT| CLOSEST_DEFENDER|CLOSEST_DEFENDER_PLAYER_ID|CLOSE_DEF_DIST|FGM|PTS|  player_name|player_id|
+--------+--------------------+--------+---+------------+-----------+------+-------------------+----------+--------+----------+---------+--------+-----------+-----------------+--------------------------+--------------+---+---+-------------+---------+
|21400899|MAR 04, 2015 - CH...|       A|  W|          24|          1|     1|2024-04-07 01:09:00|      10.8|       2|       1.9|      7.7|       2|          1|   Anderson, Alan|                    101187|           1.3|  1|  2|brian roberts|   2031

In [50]:
vector_Asmblr = VectorAssembler(inputCols=features, outputCol="features")

nba_df_vec = vector_Asmblr.transform(nba_df).select('player_name', 'features', col('SHOT_RESULT').cast('float').alias('SHOT_RESULT'))
nba_df_vec.show()

+-------------+--------------------+-----------+
|  player_name|            features|SHOT_RESULT|
+-------------+--------------------+-----------+
|brian roberts|[7.69999980926513...|        1.0|
|brian roberts|[28.2000007629394...|        0.0|
|brian roberts|[17.2000007629394...|        0.0|
|brian roberts|[3.70000004768371...|        0.0|
|brian roberts|[18.3999996185302...|        0.0|
|brian roberts|[20.7000007629394...|        0.0|
|brian roberts|[3.5,2.0999999046...|        1.0|
|brian roberts|[24.6000003814697...|        0.0|
|brian roberts|[22.3999996185302...|        0.0|
|brian roberts|[24.5,4.699999809...|        0.0|
|brian roberts|[14.6000003814697...|        1.0|
|brian roberts|[5.90000009536743...|        1.0|
|brian roberts|[26.3999996185302...|        0.0|
|brian roberts|[22.7999992370605...|        0.0|
|brian roberts|[24.7000007629394...|        1.0|
|brian roberts|[25.0,5.400000095...|        0.0|
|brian roberts|[25.6000003814697...|        0.0|
|brian roberts|[24.2

In [51]:
# above output only show 'Brian Roberts', although it has all the other players (just displayed top 20)

In [52]:
nba_K_means = KMeans(k=4, seed=1, featuresCol="features")# Having four clusters since we have 4 comfort zones to determine
nba_K_model = nba_K_means.fit(nba_df_vec)
players = ['james harden', 'chris paul', 'stephen curry', 'lebron james']# players which are mentioned in the question
nba_df_vec = nba_df_vec[nba_df_vec['player_name'].isin(players)]

In [53]:
nba_df_pred = nba_K_model.transform(nba_df_vec).select('player_name', 'prediction', 'SHOT_RESULT')

In [54]:
# Converting Spark dataframe to Pandas dataframe
# This is done so that local computations could be done quite easily
nba_df_pred_pd = nba_df_pred.toPandas()
nba_df_pred_pd = nba_df_pred_pd[nba_df_pred_pd['player_name'].isin(players)]

In [55]:
# Grouping the dataframe by player_name and prediction, calculating the mean of SHOT_RESULT for each group,
# and sorting the result by player_name and prediction
nba_result = nba_df_pred.groupBy("player_name", "prediction").mean("SHOT_RESULT").sort("player_name", "prediction")

# Adding a new column 'maxavg' that contains the maximum average SHOT_RESULT for each player_name
# using a Window function to partition by player_name
best_zone = nba_result.withColumn('maxavg', pysf.max('avg(SHOT_RESULT)').over(Window.partitionBy("player_name"))) \
                .where(pysf.col('avg(SHOT_RESULT)')==pysf.col('maxavg')).drop('maxavg').show()

best_zone

+-------------+----------+------------------+
|  player_name|prediction|  avg(SHOT_RESULT)|
+-------------+----------+------------------+
|   chris paul|         0|0.5563380281690141|
| james harden|         3|0.5604395604395604|
| lebron james|         3|0.6613545816733067|
|stephen curry|         3|0.6350710900473934|
+-------------+----------+------------------+



The prediction column indicates the Zone #

In [56]:
spark.stop()